In [31]:
import re
import pickle
import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay
from scipy.sparse import vstack, hstack, save_npz, csr_matrix
from sklearn.preprocessing import LabelEncoder

In [13]:
datafake = pd.read_csv("datafake_cleaned.csv")
datafake


,title,text,label,text_len
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1,5049
1,NaN,Did they post their votes for Hillary already?,1,46
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1,216
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0,8010
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1,1916
...,...,...,...,...
72129,Russians steal research on Trump in hack of U....,WASHINGTON (Reuters) - Hackers believed to be ...,0,4788
72130,WATCH: Giuliani Demands That Democrats Apolog...,"You know, because in fantasyland Republicans n...",1,3634
72131,Migrants Refuse To Leave Train At Refugee Camp...,Migrants Refuse To Leave Train At Refugee Camp...,0,2864
72132,Trump tussle gives unpopular Mexican leader mu...,MEXICO CITY (Reuters) - Donald Trump’s combati...,0,3374


## TF-IDF

In [14]:
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/arinaafanasieva/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/arinaafanasieva/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/arinaafanasieva/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

стопслова и лемматизация
делаем пред подготовку

In [15]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [16]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t)
        for t in tokens
        if t not in stop_words and len(t)>2
    ]
    return " ".join(tokens)

In [17]:
datafake_tfidf = datafake.copy()
datafake_tfidf["full_text"] = (datafake_tfidf["title"].astype(str) + " " + datafake_tfidf["text"].astype(str)).str.strip()
datafake_tfidf["processed_text"] = datafake_tfidf["full_text"].apply(preprocess)

сплитуем, сохраняя пропорции классов

In [18]:
x_train, x_test, y_train, y_test = train_test_split(datafake_tfidf["processed_text"],datafake_tfidf["label"], test_size=0.2, random_state=67,stratify=datafake_tfidf["label"])

In [19]:
tfidf = TfidfVectorizer(max_features=50000,ngram_range=(1, 2),min_df=2,max_df=0.9,sublinear_tf=True)

In [20]:
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)

In [21]:
x_train_tfidf.shape

(57707, 50000)

In [22]:
x_test_tfidf.shape

(14427, 50000)

добавляем ответ - лейбл чтобы легче обучать для классификации

In [24]:
train_full = hstack([x_train_tfidf, csr_matrix(y_train.values.reshape(-1, 1))])
save_npz("x_train_tfidf.npz", train_full)

In [25]:
test_full = hstack([x_test_tfidf, csr_matrix(y_test.values.reshape(-1, 1))])
save_npz("x_test_tfidf.npz", test_full)

In [26]:
train_full.shape

(57707, 50001)

In [27]:
test_full.shape

(14427, 50001)

## tf-idf для категорий

In [28]:
datacat = pd.read_json("News_Category_Dataset_v3.json", lines=True)
datacat

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22
...,...,...,...,...,...,...
209522,https://www.huffingtonpost.com/entry/rim-ceo-t...,RIM CEO Thorsten Heins' 'Significant' Plans Fo...,TECH,Verizon Wireless and AT&T are already promotin...,"Reuters, Reuters",2012-01-28
209523,https://www.huffingtonpost.com/entry/maria-sha...,Maria Sharapova Stunned By Victoria Azarenka I...,SPORTS,"Afterward, Azarenka, more effusive with the pr...",,2012-01-28
209524,https://www.huffingtonpost.com/entry/super-bow...,"Giants Over Patriots, Jets Over Colts Among M...",SPORTS,"Leading up to Super Bowl XLVI, the most talked...",,2012-01-28
209525,https://www.huffingtonpost.com/entry/aldon-smi...,Aldon Smith Arrested: 49ers Linebacker Busted ...,SPORTS,CORRECTION: An earlier version of this story i...,,2012-01-28


In [29]:
datacat_tfidf = datacat.copy()
datacat_tfidf["full_text"] = (datacat_tfidf["headline"].astype(str) + " " + datacat_tfidf["short_description"].astype(str)).str.strip()
datacat_tfidf["processed_text"] = datacat_tfidf["full_text"].apply(preprocess)

In [32]:
le = LabelEncoder()
datacat_tfidf["label"] = le.fit_transform(datacat_tfidf["category"])

In [33]:
x_train, x_test, y_train, y_test = train_test_split(datacat_tfidf["processed_text"], datacat_tfidf["label"],test_size=0.2, random_state=67, stratify=datacat_tfidf["label"])

In [ ]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True)

In [35]:
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)

In [36]:
x_train_tfidf.shape

(167621, 50000)

In [37]:
x_test_tfidf.shape

(41906, 50000)

In [38]:
train_full = hstack([x_train_tfidf, csr_matrix(y_train.values.reshape(-1, 1))])
save_npz("x_train_cat_tfidf.npz", train_full)

In [39]:
test_full = hstack([x_test_tfidf, csr_matrix(y_test.values.reshape(-1, 1))])
save_npz("x_test_cat_tfidf.npz", test_full)

In [40]:
le.classes_

array(['ARTS', 'ARTS & CULTURE', 'BLACK VOICES', 'BUSINESS', 'COLLEGE',
       'COMEDY', 'CRIME', 'CULTURE & ARTS', 'DIVORCE', 'EDUCATION',
       'ENTERTAINMENT', 'ENVIRONMENT', 'FIFTY', 'FOOD & DRINK',
       'GOOD NEWS', 'GREEN', 'HEALTHY LIVING', 'HOME & LIVING', 'IMPACT',
       'LATINO VOICES', 'MEDIA', 'MONEY', 'PARENTING', 'PARENTS',
       'POLITICS', 'QUEER VOICES', 'RELIGION', 'SCIENCE', 'SPORTS',
       'STYLE', 'STYLE & BEAUTY', 'TASTE', 'TECH', 'THE WORLDPOST',
       'TRAVEL', 'U.S. NEWS', 'WEDDINGS', 'WEIRD NEWS', 'WELLNESS',
       'WOMEN', 'WORLD NEWS', 'WORLDPOST'], dtype=object)

итог: 2 датасета tf-idf, векторы (эмбединги новостей) и последняя колонка - лейбл (категория)
применен label encoder